# Credit Card Fraud — EDA
**ML Drift Monitor · Milestone M1**

Goals:
1. Understand the class distribution and severity of imbalance
2. Explore feature distributions (V1–V28, Amount, Time)
3. Identify features most correlated with fraud
4. Validate the rationale for SMOTE vs. other balancing strategies
5. Understand which features are most likely to drift (V4, V11, Amount)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = '../data/raw/creditcard.csv'

In [ ]:
# Load dataset
if not Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f'{DATA_PATH} not found. '
        'Run: kaggle datasets download -d mlg-ulb/creditcardfraud -p ../data/raw --unzip'
    )

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean()

print(f'Legitimate transactions: {class_counts[0]:,}  ({1-fraud_rate:.4%})')
print(f'Fraud transactions:      {class_counts[1]:,}  ({fraud_rate:.4%})')
print(f'Imbalance ratio:         {class_counts[0]/class_counts[1]:.0f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Absolute counts
axes[0].bar(['Legitimate', 'Fraud'], class_counts.values, color=['#2196F3', '#F44336'])
axes[0].set_title('Transaction Counts')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Percentage
axes[1].pie(
    [1 - fraud_rate, fraud_rate],
    labels=['Legitimate', 'Fraud'],
    colors=['#2196F3', '#F44336'],
    autopct='%1.3f%%',
    startangle=90,
)
axes[1].set_title('Class Distribution')

plt.tight_layout()
plt.show()

print('\n⚠️  Extreme imbalance: naive classifier that predicts all-legitimate gets 99.83% accuracy.')
print('   This is why we use F1, AUC-ROC, and Precision-Recall curve as primary metrics.')

## 2. Feature Overview

In [ ]:
print('Dataset info:')
print(f'  Features: {list(df.columns)}')
print(f'  Missing values: {df.isnull().sum().sum()}')
print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
df.describe()

In [ ]:
# Distribution of Amount by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

fraud = df[df['Class'] == 1]['Amount']
legit = df[df['Class'] == 0]['Amount']

axes[0].hist([legit, fraud], bins=50, label=['Legitimate', 'Fraud'],
             color=['#2196F3', '#F44336'], alpha=0.7)
axes[0].set_yscale('log')
axes[0].set_title('Transaction Amount Distribution (log scale)')
axes[0].set_xlabel('Amount (€)')
axes[0].legend()

axes[1].boxplot([legit.clip(0, 500), fraud.clip(0, 500)],
                labels=['Legitimate', 'Fraud'], patch_artist=True,
                boxprops=dict(facecolor='#E3F2FD'),
                medianprops=dict(color='#1976D2', linewidth=2))
axes[1].set_title('Amount Boxplot (clipped at 500)')
axes[1].set_ylabel('Amount (€)')

plt.tight_layout()
plt.show()

print(f'Median fraud amount: €{fraud.median():.2f}')
print(f'Median legit amount: €{legit.median():.2f}')

## 3. Feature Correlation with Fraud

In [ ]:
# Point-biserial correlation of each feature with the fraud label
feature_cols = [c for c in df.columns if c != 'Class']
correlations = df[feature_cols].corrwith(df['Class']).sort_values(key=abs, ascending=False)

plt.figure(figsize=(14, 5))
colors = ['#F44336' if v < 0 else '#4CAF50' for v in correlations]
plt.bar(correlations.index, correlations.values, color=colors, alpha=0.8)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Fraud Label')
plt.ylabel('Correlation coefficient')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Top 5 features correlated with fraud:')
print(correlations.head())

## 4. Drift-Target Features: V4, V11, Amount

In [ ]:
# Justify why V4, V11, and Amount are chosen for drift simulation
drift_features = ['V4', 'V11', 'Amount']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, feat in zip(axes, drift_features):
    fraud_vals = df[df['Class'] == 1][feat]
    legit_vals = df[df['Class'] == 0][feat]
    
    ax.hist(legit_vals, bins=50, alpha=0.6, label='Legitimate', color='#2196F3', density=True)
    ax.hist(fraud_vals, bins=50, alpha=0.6, label='Fraud', color='#F44336', density=True)
    ax.set_title(f'{feat} — Fraud vs Legitimate')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Drift Target Features: Distinct Distributions Between Classes', y=1.02)
plt.tight_layout()
plt.show()

print('\nRationale for choosing these features for drift simulation:')
print('  V4  : High correlation with fraud; clear bimodal separation')
print('  V11 : Negative correlation with fraud; different skew between classes')
print('  Amount: Raw feature (not PCA); most interpretable drift to demonstrate')

## 5. Imbalance Handling Strategy

In [ ]:
print('Imbalance handling strategies compared:\n')

strategies = {
    'No handling': 'Biased toward majority class; near-zero recall on fraud',
    'Class weights (scale_pos_weight)': 'Fast, no data generation. May under-represent minority patterns',
    'Undersampling': 'Loses majority class information. Risky with only 492 fraud cases',
    'SMOTE (chosen)': 'Generates synthetic minority samples → balanced training. Preserves all data',
    'SMOTE + class weights': 'Redundant — pick one; combining can over-correct',
}

for strategy, note in strategies.items():
    marker = '✅' if 'chosen' in strategy else '  '
    print(f'{marker} {strategy:40s} → {note}')

print('\nDecision: SMOTE preserves majority class patterns and generates diverse fraud examples.')
print('  XGBoost scale_pos_weight is set as a fallback hedge; SMOTE is the primary mechanism.')

## 6. Correlation Matrix (Top Features)

In [ ]:
top_features = correlations.head(10).index.tolist() + ['Class']
corr_matrix = df[top_features].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
)
plt.title('Correlation Matrix — Top 10 Features + Class')
plt.tight_layout()
plt.show()

## 7. Summary

| Finding | Implication |
|---|---|
| 577:1 class imbalance | Use SMOTE + F1/PR-AUC as metrics, not accuracy |
| V4, V11, Amount have distinct fraud distributions | Good candidates for drift simulation |
| PCA features V1–V28 are uncorrelated by design | Simplifies drift: shift one without affecting others |
| No missing values | No imputation needed |
| Amount is right-skewed | Consider log-transform for Amount in production features |

**Next step:** `make train` — XGBoost with SMOTE targeting F1 > 0.85